In [ ]:
import pandas as pd
from datetime import *
import os 
import numpy as np
from scipy.signal import find_peaks as fp

# Constants
global WORKING_DIRECTORY,RAWDATAFILES,SUPPORTED_FILEFORMATS,RAW_Data_DIR,MERGE_FILES, LABEL_DF

SUPPORTED_FILEFORMATS={"CSV":pd.read_csv,"TXT":pd.read_fwf,"JSON":pd.read_json,"XML":pd.read_xml,"XLSX":pd.read_excel,"XLS":pd.read_excel}
COL_Prefixes ={"HeartRate":"HR_","Metadata":"MD_","WatchAccelerometer":"ACC_", "WatchGravity":"G_","WatchGyroscope":"GYRO_","WatchOrientation":"OR_","WatchTotalAcceleration":"TACC_","WatchMagnetometer":"MAG_"}

RAWDATAFILES={}

MERGE_FILES=1

pd.set_option('display.max_rows', 500)

# Get the current working directory
cwd = os.getcwd()
WORKING_DIRECTORY = os.path.dirname(os.path.dirname(cwd)) #Parent directory
print("The current working directory is:", WORKING_DIRECTORY)

DATA_DIRECTORY = os.path.join(WORKING_DIRECTORY,"Project\\Data")
print("The data directory is:", DATA_DIRECTORY)

RAW_Data_DIR=  os.path.join(DATA_DIRECTORY,"01-raw")
PREPROCESS_Data_DIR=  os.path.join(DATA_DIRECTORY,"02-preprocessed")
FEATURES_Data_DIR=  os.path.join(DATA_DIRECTORY,"03-features")
PREDICTIONS_Data_DIR=  os.path.join(DATA_DIRECTORY,"04-predictions")

#import previous modules output
PREPROC_DF = pd.read_pickle(os.path.join(PREPROCESS_Data_DIR,"250MS_Resampled.pkl"))
TRAIN_DF= pd.read_excel(os.path.join(RAW_Data_DIR,"Training Notes.Xlsx"))

TRAIN_DF.dropna(subset=["Time"],inplace=True)

print(PREPROC_DF.shape)

In [ ]:
#Clean Up
dropcols = [v  for v in PREPROC_DF.columns if (v.find("seconds")>-1 or v.find("time")>-1)]
PREPROC_DF.drop(columns=dropcols,inplace=True)
PREPROC_DF.dropna(how='all',inplace=True)

#interpolate quadratic
PREPROC_DF.interpolate(method='quadratic',axis=0,inplace=True)
print(PREPROC_DF.isna().sum())
print(PREPROC_DF.shape)

In [ ]:
#Translate Training Notes to Y Values
LABEL_DF=pd.DataFrame(columns=["time_start","time_end","target","target_name","n_repetition","weight_kg"])

#sampling timespan 250ms
sam_ts = timedelta(0,0.25)

#Average timespan of a set 12*3 seconds
set_ts = timedelta(0,36)

#Average timespan of break 2min
break_ts = timedelta(0,120)

#Average timespan of switching exercise
switch_ts = timedelta(0,60)

offset_ts = "00:00:00"

#Set of Exercises
exercise_set = list((x.strip() for x in TRAIN_DF["exercise"]))
exercise_set = set(exercise_set)
exercise_set = list(exercise_set)
exercise_set.sort()
exercise_set = ["break","switch"] + exercise_set
exercise_dict = dict([(idx, val) for idx, val in enumerate(exercise_set)])


#itterate through exercises
for index, row in TRAIN_DF.iterrows():
    reps = row["repetitions"].split(",")
    reps= [int(x) for x in reps]
    if type(row["weight_kg"]) is not str or str(row["weight_kg"]).find(",")==-1:       
        weights = [float(row["weight_kg"])] * len(reps)
    else:
        weights = row["weight_kg"].split(",")
        weights = [float(x) for x in weights]

    if len(str(row["weight_kg"])) ==0:
        weights=[np.nan] * len(reps)
        
    dt = pd.to_datetime(row['Date'] + " " + row['Time'], format="%d.%m.%Y %H:%M")

    #iterate trhough sets
    for idx, rep in enumerate(reps):
        key, = [k for k, v in exercise_dict.items() if v == row["exercise"].strip()]
        if (len(weights)>=idx):
            weight= weights[idx]

        LABEL_DF.loc[len(LABEL_DF)] = [dt,dt + set_ts -sam_ts , key, row["exercise"].strip(), rep, weight]
        dt=dt + set_ts

        if (idx != len(reps)-1):
            LABEL_DF.loc[len(LABEL_DF)] = [dt,dt + break_ts -sam_ts,  0, "break", np.nan, np.nan]
            dt=dt + break_ts
        else:
            LABEL_DF.loc[len(LABEL_DF)] = [dt,dt + switch_ts -sam_ts,  1, "switch", np.nan, np.nan]
            dt=dt+ switch_ts

In [ ]:
LABEL_DF

In [ ]:
#Boundary Refinement
if "is_Y" not in LABEL_DF.columns:
    LABEL_DF["is_Y"] =np.nan

def refine_set(setindex,  new_ts_start:datetime = pd.NaT , new_ts_end: datetime = pd.NaT , is_Y = 1,adjust_adjacent=1):
    global LABEL_DF
    sam_ts = timedelta(0,0.25)

    if new_ts_start is pd.NaT:
        new_ts_start = LABEL_DF.loc[setindex,"time_start"]
   # else: new_ts_start = #pd.to_datetime(new_ts_start,"%Y-%m-%d %H:%M:%S.%f")
    if new_ts_end is pd.NaT:
        new_ts_end = LABEL_DF.loc[setindex,"time_end"]
    #else: new_ts_end = pd.to_datetime(new_ts_end,format="%Y-%m-%d %H:%M:%S.%f")
    LABEL_DF.loc[setindex] = [new_ts_start,new_ts_end, LABEL_DF.loc[setindex,"target"], LABEL_DF.loc[setindex,"target_name"], LABEL_DF.loc[setindex,"n_repetition"], LABEL_DF.loc[setindex,"weight_kg"],is_Y]
    if adjust_adjacent==1:
        LABEL_DF.loc[setindex-1,"time_end"] = pd.to_datetime(new_ts_start) - sam_ts
        LABEL_DF.loc[setindex+1,"time_start"] = pd.to_datetime(new_ts_end) + sam_ts



In [ ]:
#Split Set into multiple, due to breaks within set
def split_set(setindex,n_split):
    global LABEL_DF
    if n_split<2:
        return

    df=LABEL_DF.copy()
    new_row_df = pd.DataFrame([LABEL_DF.loc[setindex].values],columns=LABEL_DF.columns)

    for i in range(1,n_split):
        #LABEL_DF.loc[setindex+i] = LABEL_DF.loc[setindex]
        df = pd.concat([df.iloc[:setindex], new_row_df, df.iloc[setindex:]], ignore_index=True)

    LABEL_DF = df.copy()

In [ ]:
def save_label():
    global LABEL_DF
    LABEL_DF.to_pickle(os.path.join(PREPROCESS_Data_DIR,"label.pkl"))
def load_label():
    global LABEL_DF
    LABEL_DF = pd.read_pickle(os.path.join(PREPROCESS_Data_DIR,"label.pkl"))
LABEL_DF.info()

In [ ]:
#generate template
def generatetemplate(row_start):
    for row in LABEL_DF.itertuples():
        if ((row[0]>row_start) & (row[0] % 2 != 0)):
            print(f"refine_set({row[0]},\"{row[1]}\", \"{row[2]}\")")


In [ ]:
#Refine Boundaries
refine_set(42,"2025-12-12 14:48:30", "2025-12-12 14:49:10")
refine_set(44,"2025-12-12 14:51:33.75", "2025-12-12 14:52:14")
refine_set(46,"2025-12-12 14:55:20", "2025-12-12 14:55:58.25")
refine_set(54,"2025-12-12 15:15:38", "2025-12-12 15:16:33.5")
refine_set(56,"2025-12-12 15:19:15", "2025-12-12 15:20:12.25")
refine_set(58,"2025-12-12 15:20:34.5", "2025-12-12 15:22:09.5")
refine_set(148,"2025-12-16 13:00:19", "2025-12-16 13:00:51")
refine_set(150,"2025-12-16 13:03:39", "2025-12-16 13:04:07")
refine_set(152,"2025-12-16 13:06:34.5", "2025-12-16 13:07:10")
refine_set(226,"2025-12-19 14:55:38.5", "2025-12-19 14:55:52.25")
refine_set(262,"2025-12-19 15:02:22.5", "2025-12-19 15:02:55.75")
refine_set(264,"2025-12-19 15:05:31.75", "2025-12-19 15:06:09.75")
refine_set(266,"2025-12-19 15:08:43", "2025-12-19 15:09:27")
refine_set(270,"2025-12-19 15:15:47.5", "2025-12-19 15:17:01")
refine_set(272,"2025-12-19 15:19:27", "2025-12-19 15:19:40.75")
split_set(272,4)
refine_set(273,"2025-12-19 15:19:46", "2025-12-19 15:20:07.75",adjust_adjacent=0)
refine_set(274,"2025-12-19 15:20:27.5", "2025-12-19 15:20:51.25",adjust_adjacent=0)
refine_set(275,"2025-12-19 15:21:07", "2025-12-19 15:21:42.5",adjust_adjacent=0)
refine_set(277,"2025-12-22 11:08:50", "2025-12-22 11:09:28.25")
refine_set(279,"2025-12-22 11:12:15", "2025-12-22 11:12:54.5")
refine_set(281,"2025-12-22 11:16:08", "2025-12-22 11:16:53.25")
refine_set(283,"2025-12-22 11:21:08", "2025-12-22 11:21:28")
refine_set(285,"2025-12-22 11:23:48.75", "2025-12-22 11:24:30")
refine_set(287,"2025-12-22 11:26:12.25", "2025-12-22 11:26:56")
refine_set(289,"2025-12-22 11:33:00.25", "2025-12-22 11:33:33.75")
refine_set(291,"2025-12-22 11:35:59", "2025-12-22 11:36:32")
refine_set(293,"2025-12-22 11:38:42", "2025-12-22 11:39:05.75")

refine_set(295,"2025-12-22 11:43:18.75", "2025-12-22 11:43:52.75")
refine_set(297,"2025-12-22 11:44:55.75", "2025-12-22 11:45:25.25")
refine_set(299,"2025-12-22 11:46:46.5", "2025-12-22 11:47:18.5")

refine_set(301,"2025-12-22 11:52:22.5", "2025-12-22 11:53:02.25")
refine_set(303,"2025-12-22 11:54:57.25", "2025-12-22 11:55:33.5")
#3rd set not found

refine_set(307,"2025-12-22 12:02:51", "2025-12-22 12:03:10.25")
refine_set(309,"2025-12-22 12:04:59", "2025-12-22 12:05:21.5")
refine_set(311,"2025-12-22 12:07:09", "2025-12-22 12:07:40")

refine_set(313,"2025-12-22 12:13:09", "2025-12-22 12:13:50.75")
refine_set(315,"2025-12-22 12:15:59", "2025-12-22 12:16:34")
#3rd set not found

refine_set(319,"2025-12-22 12:22:59", "2025-12-22 12:23:39")
refine_set(321,"2025-12-22 12:25:57.5", "2025-12-22 12:26:31")
#3rd set not found

refine_set(325,"2025-12-23 09:33:44", "2025-12-23 09:34:21.5")
refine_set(327,"2025-12-23 09:38:02", "2025-12-23 09:38:29.75")
refine_set(329,"2025-12-23 09:41:45", "2025-12-23 09:42:24.25")

refine_set(331,"2025-12-23 09:45:12", "2025-12-23 09:45:43.75")
refine_set(333,"2025-12-23 09:47:47", "2025-12-23 09:48:13")
refine_set(335,"2025-12-23 09:50:14", "2025-12-23 09:50:44")

refine_set(337,"2025-12-23 09:54:36", "2025-12-23 09:55:01")
refine_set(339,"2025-12-23 09:57:09", "2025-12-23 09:57:40")
refine_set(341,"2025-12-23 10:01:03.25", "2025-12-23 10:01:36.75")

refine_set(343,"2025-12-23 10:05:22.25", "2025-12-23 10:05:54")
refine_set(345,"2025-12-23 10:07:56.25", "2025-12-23 10:08:33")
refine_set(347,"2025-12-23 10:11:41.25", "2025-12-23 10:12:13")

refine_set(349,"2025-12-23 10:14:18.25", "2025-12-23 10:14:44")
refine_set(351,"2025-12-23 10:16:54.25", "2025-12-23 10:17:19")
refine_set(353,"2025-12-23 10:20:10.25", "2025-12-23 10:20:34")

refine_set(355,"2025-12-23 10:23:23", "2025-12-23 10:23:52")
refine_set(357,"2025-12-23 10:27:16", "2025-12-23 10:27:43")
refine_set(359,"2025-12-23 10:30:34", "2025-12-23 10:31:03")

refine_set(361,"2025-12-23 10:33:47", "2025-12-23 10:34:14")
refine_set(363,"2025-12-23 10:36:59", "2025-12-23 10:37:27")
refine_set(365,"2025-12-23 10:39:49", "2025-12-23 10:40:13")

refine_set(367,"2025-12-23 10:41:45", "2025-12-23 10:42:04")
refine_set(369,"2025-12-23 10:43:41", "2025-12-23 10:44:15")
refine_set(371,"2025-12-23 10:46:23", "2025-12-23 10:47:07")

refine_set(373,"2025-12-23 10:48:34", "2025-12-23 10:49:09")
refine_set(375,"2025-12-23 10:51:17", "2025-12-23 10:51:52")
refine_set(377,"2025-12-23 10:53:53", "2025-12-23 10:54:30")

refine_set(379,"2025-12-23 10:58:54", "2025-12-23 10:59:33")
refine_set(381,"2025-12-23 11:01:32", "2025-12-23 11:02:19")
refine_set(383,"2025-12-23 11:04:32", "2025-12-23 11:05:18")

refine_set(385,"2025-12-23 11:08:57", "2025-12-23 11:09:45")
refine_set(387,"2025-12-23 11:11:55", "2025-12-23 11:12:33")
refine_set(389,"2025-12-23 11:15:02", "2025-12-23 11:15:49")

refine_set(391,"2025-12-23 11:18:13", "2025-12-23 11:19:13")
refine_set(393,"2025-12-23 11:22:03", "2025-12-23 11:23:05")
refine_set(395,"2025-12-23 11:25:45", "2025-12-23 11:26:52")

refine_set(397,"2025-12-26 10:58:25.5", "2025-12-26 10:58:56.25")
refine_set(399,"2025-12-26 11:01:40", "2025-12-26 11:02:15")
refine_set(401,"2025-12-26 11:06:27", "2025-12-26 11:07:07")
refine_set(403,"2025-12-26 11:12:19.5", "2025-12-26 11:12:59")

refine_set(405,"2025-12-26 11:16:02.5", "2025-12-26 11:17:03")
refine_set(407,"2025-12-26 11:19:12", "2025-12-26 11:20:23.75")
refine_set(409,"2025-12-26 11:22:53", "2025-12-26 11:23:40.75")

refine_set(411,"2025-12-26 11:29:35", "2025-12-26 11:30:18.75")
refine_set(413,"2025-12-26 11:32:25", "2025-12-26 11:33:15.75")
refine_set(415,"2025-12-26 11:35:47", "2025-12-26 11:36:28.75")

refine_set(417,"2025-12-26 11:42:56", "2025-12-26 11:43:25.75")
refine_set(419,"2025-12-26 11:45:25", "2025-12-26 11:46:27.5")
refine_set(421,"2025-12-26 11:49:47", "2025-12-26 11:50:37.75")

refine_set(423,"2025-12-26 11:54:03", "2025-12-26 11:54:35.75")
refine_set(425,"2025-12-26 11:56:44", "2025-12-26 11:57:11.25")
refine_set(427,"2025-12-26 11:59:42", "2025-12-26 12:00:24.75")

refine_set(429,"2025-12-26 12:03:08", "2025-12-26 12:03:52.75")
refine_set(431,"2025-12-26 12:06:30", "2025-12-26 12:07:25.75")
refine_set(433,"2025-12-26 12:10:22", "2025-12-26 12:10:56")

refine_set(435,"2025-12-26 12:13:23", "2025-12-26 12:13:55.75")
refine_set(437,"2025-12-26 12:16:15", "2025-12-26 12:16:43.75")
refine_set(439,"2025-12-26 12:19:04", "2025-12-26 12:19:26.75")

#todo
refine_set(441,"2025-12-27 15:53:49", "2025-12-27 15:54:14.75")
refine_set(443,"2025-12-27 15:56:27", "2025-12-27 15:56:51.75")
refine_set(445,"2025-12-27 16:00:02", "2025-12-27 16:00:38.75")

refine_set(447,"2025-12-27 16:04:11", "2025-12-27 16:04:43.75")
refine_set(449,"2025-12-27 16:06:51", "2025-12-27 16:07:22.75")
refine_set(451,"2025-12-27 16:09:44", "2025-12-27 16:10:12.75")

refine_set(453,"2025-12-27 16:12:56", "2025-12-27 16:13:32.75")
refine_set(455,"2025-12-27 16:16:55", "2025-12-27 16:17:32.75")
refine_set(457,"2025-12-27 16:19:47", "2025-12-27 16:20:22")

refine_set(459,"2025-12-27 16:24:02.5", "2025-12-27 16:24:27.75")
refine_set(461,"2025-12-27 16:26:25", "2025-12-27 16:26:51.75")
refine_set(463,"2025-12-27 16:29:38", "2025-12-27 16:30:04.75")

refine_set(465,"2025-12-27 16:32:12", "2025-12-27 16:32:49.75")
refine_set(467,"2025-12-27 16:35:22", "2025-12-27 16:35:56.75")
refine_set(469,"2025-12-27 16:38:10", "2025-12-27 16:38:37.75")

refine_set(471,"2025-12-27 16:41:09", "2025-12-27 16:41:44.75")
refine_set(473,"2025-12-27 16:44:35", "2025-12-27 16:45:10")
refine_set(475,"2025-12-27 16:47:23", "2025-12-27 16:47:59")

refine_set(477,"2025-12-27 16:51:56", "2025-12-27 16:52:41.75")
refine_set(479,"2025-12-27 16:55:41", "2025-12-27 16:56:31.75")
refine_set(481,"2025-12-27 16:58:44", "2025-12-27 16:59:22.75")

refine_set(483,"2025-12-27 17:01:06", "2025-12-27 17:01:27.75")
refine_set(485,"2025-12-27 17:05:25", "2025-12-27 17:05:53")
refine_set(487,"2025-12-27 17:09:00", "2025-12-27 17:09:29")

refine_set(489,"2025-12-27 17:13:07", "2025-12-27 17:13:39")
refine_set(491,"2025-12-27 17:17:40", "2025-12-27 17:18:10")
refine_set(493,"2025-12-27 17:21:13", "2025-12-27 17:21:47.75")

refine_set(495,"2025-12-27 17:23:28", "2025-12-27 17:24:07.75")
refine_set(497,"2025-12-27 17:26:47.75", "2025-12-27 17:27:29")
refine_set(499,"2025-12-27 17:29:56", "2025-12-27 17:30:39.75")

refine_set(501,"2025-12-27 17:35:50", "2025-12-27 17:37:09.25")
refine_set(503,"2025-12-27 17:39:35", "2025-12-27 17:40:45")
refine_set(505,"2025-12-27 17:43:39.5", "2025-12-27 17:44:34.75")
split_set(505,2)
refine_set(506,"2025-12-27 17:45:30", "2025-12-27 17:46:36.75",adjust_adjacent=0)





In [ ]:
#2nd Iteration
#constrain window to the actual execution of the exercise
def constrain_window():
    global LABEL_DF
    time_start_delta = timedelta(0,1)
    time_end_delta = timedelta(0,.5)

    for idx, row in LABEL_DF.iterrows():
        
        if ((row["target"] <= 1)| (idx==0)):
            continue
        
        if (LABEL_DF.loc[idx-1, "target"]<=1):
            if ((LABEL_DF.loc[idx-1, "target"] != LABEL_DF.loc[idx, "target"])):
                refine_set(idx,row["time_start"]+time_start_delta , row["time_end"]-time_end_delta)

constrain_window()

In [ ]:
#Labelling
for label in LABEL_DF.itertuples():
    # #target columns in PREPROC
    PREPROC_DF.loc[label[1]:label[2], "target"] = int(label[3])
    PREPROC_DF.loc[label[1]:label[2],"target_name"] = label[4]
    PREPROC_DF.loc[label[1]:label[2],"target_repetitions"] = np.absolute(label[5])
    PREPROC_DF.loc[label[1]:label[2],"target_weight"] = label[6]
    PREPROC_DF.loc[label[1]:label[2],"is_target"] = label[7]

PREPROC_DF.is_target = PREPROC_DF.is_target.fillna(0)

In [ ]:
#export data for next steps
PREPROC_DF.to_pickle(os.path.join(FEATURES_Data_DIR,"250MS_labeled.pkl"))

exercise_df=pd.DataFrame(exercise_dict.items(), columns=["key","value"],)
exercise_df.to_pickle(os.path.join(FEATURES_Data_DIR,"exercise_dict.pkl"))

In [ ]:
# run lines in terminal to convert into html
print(f"cas_gpu_venv\\Scripts\\activate.bat")
print(f"jupyter-nbconvert --to html \"0_Module\\M3_Machine_Learning\\Project\\notebooks\\{os.path.basename(globals()['__vsc_ipynb_file__'])}\"")